In [0]:
from pyspark.sql import SparkSession

spark = SparkSession\
            .builder\
                .appName("Spark Transformation")\
                    .getOrCreate()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql import Row
from datetime import date

# Define the employee schema
emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", StringType(), True),
    StructField("hire_date", DateType(), True),
    StructField("country", StringType(), True),
    StructField("age", IntegerType(), True),
])

# 20 rows of sample employee data
emp_rows = [
    Row(1, "John Smith", "Engineering", 85000.0, date(2021, 3, 15), "USA", 30),
    Row(2, "Alice Johnson", "Engineering", 92000.0, date(2020, 7, 1), "USA", 35),
    Row(3, "Bob Williams", "Sales", 65000.0, date(2022, 1, 20), "UK", 28),
    Row(4, "Carol Brown", "Sales", 68000.0, date(2019, 11, 5), "UK", 40),
    Row(5, "David Jones", "HR", 55000.0, date(2021, 9, 10), "Canada", 32),
    Row(6, "Eva Garcia", "HR", 58000.0, date(2023, 2, 28), "Canada", 26),
    Row(7, "Frank Miller", "Engineering", 78000.0, date(2018, 5, 22), "Germany", 45),
    Row(8, "Grace Davis", "Marketing", 72000.0, date(2022, 8, 14), "Germany", 29),
    Row(9, "Henry Wilson", "Marketing", 70000.0, date(2020, 12, 1), "India", 33),
    Row(10, "Ivy Taylor", "Finance", 95000.0, date(2017, 6, 30), "India", 50),
    Row(11, "Jack Anderson", "Finance", 89000.0, date(2021, 4, 18), "Australia", 38),
    Row(12, "Karen Thomas", "Engineering", 81000.0, date(2023, 3, 7), "Australia", 25),
    Row(13, "Leo Moore", "Sales", 62000.0, date(2019, 10, 25), "USA", 37),
    Row(14, "Mia Martin", "Engineering", 90000.0, date(2022, 5, 11), "USA", 27),
    Row(15, "Nick Lee", "HR", 60000.0, date(2018, 2, 9), "UK", 42),
    Row(16, "Olivia Harris", "Marketing", 74000.0, date(2020, 9, 3), "Canada", 31),
    Row(17, "Paul Clark", "Finance", 87000.0, date(2023, 7, 19), "Germany", 24),
    Row(18, "Quinn Lewis", "Engineering", 83000.0, date(2021, 12, 12), "India", 34),
    Row(19, "Rachel Walker", "Sales", 66000.0, date(2019, 8, 8), "Australia", 36),
    Row(20, "Sam Hall", "Marketing", 71000.0, date(2022, 11, 27), "USA", 23),
]

In [0]:
emp = spark.\
    createDataFrame(emp_rows, schema=emp_schema)

emp.printSchema()

In [0]:
## Column transfomation
# select emp_id, name, cast salary to double, age
from pyspark.sql.functions import expr, col

emp_casted = emp.\
                select(
                    'emp_id', 'emp_name', 'age',
                    expr('cast(salary as double) as sal'))
                
emp_cast = emp.\
              select(
                  'emp_id', 'emp_name', 'age',
                  col('salary').cast('double')
              )
                
emp_cast.printSchema()
emp_casted.printSchema()

In [0]:
## adding a column in a df
# add a tax column which will be 15% of salary
emp_taxed = emp_cast.withColumn(
                "tax", col("salary") * 0.15)

display(emp_taxed)

In [0]:
## Literals
# Select emp_id, name, salary, tax, 1 as columnOne, two as columntwo from Emp_taxted
from pyspark.sql.functions import lit

emp_new_cols = emp_taxed.\
                    withColumn("columnOne", lit(1)).\
                        withColumn('columnTwo', lit('two'))

emp_new_cols.show()

In [0]:
# renaming a column
# salary to gross_sal
emp_sal = emp_new_cols.withColumnRenamed('salary', 'gross-sal')
emp_sal.show()

#### Note: We can name a column with spaces but it is not recommendabel cuz in downstream it will throw an **Error**.

In [0]:
## removing/deleting a column
col_dropped = emp_new_cols.drop('columnTwo', 'columnOne')
display(col_dropped)

In [0]:
## filtering the data
## Advanced filtering top 5 rows with highest tax
# default orderBy is ascending order

emp_filter = col_dropped.\
                where('tax > 10000').\
                    orderBy(col("tax").desc()).\
                        limit(5)

emp_filter.show()

#### Adding multiple coulmns

For that, create a dictiuonary of columns with key and info as values and use `withColumns`.

In [0]:
columns = {
    "tax": col('salary') * 0.15,
    "columnOne": lit(1),
    "columntwo": lit('two')
}

emp_added_col = emp.withColumns(columns)
emp_added_col.show()